# Creating a keras model for jet tagging

A lot of inspiration for this guide is taken from [here](https://github.com/fastmachinelearning/hls4ml-tutorial/blob/master/https://github.com/fastmachinelearning/hls4ml-tutorial). 
It's definitily worth to check out and they provide examples and information for advanced techniques.

## Get additional packages

Check if your virtual enviroment is active!

In [ ]:
!which python

And again, creating a new project directory ...

In [ ]:
%cd tutorial-hls4ml/

Now let's get some more dependecies ...

In [ ]:
!pip install sklearn

With the additional packages installed we can import the libraries needed.

In [ ]:
from tensorflow.keras.utils import to_categorical
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
from utils import plotting
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.metrics import accuracy_score

seed = 0
np.random.seed(seed)
import tensorflow as tf
tf.random.set_seed(seed)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1
from utils.callbacks import all_callbacks

import os

Create a folder to store the dataset.

In [ ]:
if os.path.exists('data'):
    print('Folder \'data\' already created.')
else:
    print('Folder \'data\' created.')
    os.mkdir('data')

Fetch the dataset from OpenML. More information are aviable under [OpenML](https://www.openml.org/d/42468)  
This short description can also be found there.
```
Identify jets of particles from the LHC, created for the study of ultra low latency inference with hls4ml.
Use 16 high level features to identify the 5 jet classes: quark (q), gluon (g), W boson (w), Z boson (z), or top quark (t).
```




In [ ]:
data = fetch_openml('hls4ml_lhc_jets_hlf')
X, y = data['data'], data['target']

Now let's have a look at the data.

In [ ]:
print(data['feature_names'])
print(X.shape, y.shape)
print(X[:5])
print(y[:5])

Following fields split up the dataset in a train and test set. 
Afterwards they get transformed and saved in the data folder.

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y)
y = to_categorical(y, 5)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_val = scaler.fit_transform(X_train_val)
X_test = scaler.transform(X_test)

In [ ]:
np.save(os.path.join('data','X_train_val.npy'), X_train_val)
np.save(os.path.join('data','X_test.npy'), X_test)
np.save(os.path.join('data','y_train_val.npy'), y_train_val)
np.save(os.path.join('data','y_test.npy'), y_test)
np.save(os.path.join('data','classes.npy'), le.classes_)

Now it's time to create our network description. 
Consisting out of three dense (fully connected) layer with 64, 32 and 32 neurons. 
This network uses the ReLU activation in between and a softmax for the output layer. 

In [ ]:
model = Sequential()
model.add(Dense(64, input_shape=(16,), name='fc1', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu1'))
model.add(Dense(32, name='fc2', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu2'))
model.add(Dense(32, name='fc3', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu3'))
model.add(Dense(5, name='output', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='softmax', name='softmax'))

To get a nicely looking summary keras models have a build-in method. 
Which also shows the amount of parameters needed.

In [ ]:
model.summary()

The network is trained over 30 epoches and will be saved in the h5-format and json. 
This is important for the use with the HLS4ML framework.

In [ ]:
train = True
if train:
    adam = Adam(lr=0.0001)
    model.compile(optimizer=adam, loss=['categorical_crossentropy'], metrics=['accuracy'])
    callbacks = all_callbacks(stop_patience = 1000,
                              lr_factor = 0.5,
                              lr_patience = 10,
                              lr_epsilon = 0.000001,
                              lr_cooldown = 2,
                              lr_minimum = 0.0000001,
                              outputDir = 'jet_tagging_keras')
    model.fit(X_train_val, y_train_val, batch_size=1024,
              epochs=30, validation_split=0.25, shuffle=True,
              callbacks = callbacks.callbacks)

    model_json = model.to_json()
    with open("jet_tagging_keras/jet_tagging.json", "w") as json_file:
        json_file.write(model_json)
else:
    from tensorflow.keras.models import load_model
    model = load_model('jet_tagging_keras/KERAS_check_best_model.h5')

In [ ]:
y_keras = model.predict(X_test)

In [ ]:
print("Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
plt.figure(figsize=(9,9))
_ = plotting.makeRoc(y_test, y_keras, le.classes_)

### Now let's make a hls model!
Go on with part 3: 3_Convert_keras_to_hls